In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date

CATALOG = "workspace"
LANDING = f"/Volumes/{CATALOG}/bronze/landing"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/checkpoints/bronze_events"
TARGET = f"{CATALOG}.bronze.events_raw"

stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT)
    .option("cloudFiles.schemaHints", "id STRING, payload STRING")
    .load(LANDING)
    .withColumn("_ingest_file", col("_metadata.file_path"))
    .withColumn("_ingest_ts", current_timestamp())
    .withColumn("_ingest_date", to_date(col("_ingest_ts")))
)

(
    stream.writeStream
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow = True)
    .partitionBy("_ingest_date")
    .toTable(TARGET)
    .awaitTermination()
)

In [0]:
%sql
SELECT _ingest_file, COUNT(*) FROM workspace.bronze.events_raw GROUP BY 1